In [2]:
# DiD

import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS


# Load dataset

df = pd.read_stata("marriage_level.dta")


# Convert class variables

df['clbr'] = pd.to_numeric(df['clbr'], errors='coerce')
df['clgr'] = pd.to_numeric(df['clgr'], errors='coerce')


# Dependent variables

df['classdiff'] = df['clbr'] - df['clgr']

df['mardn'] = np.where(
    (df['clbr'] > df['clgr']) & (df['clbr'].notnull()),
    1, 0
)

df['lowbr'] = np.where(
    (df['clbr'] <= 7) & (df['clbr'] >= 5),
    1, 0
)


# Covariates

df['agebrd'] = df['agebr'] / 100
df['agegrd'] = df['agegr'] / 100


# DiD interaction

# If not already created:
df['post_mortality'] = df['post'] * df['mortality']


# Drop missing values

cols_needed = [
    'classdiff', 'post_mortality',
    'agebrd', 'agegrd', 'rural',
    'clgr', 'depc', 'year'
]

df_clean = df.dropna(subset=cols_needed)


# Groom class fixed effects

df_clean = pd.get_dummies(df_clean, columns=['clgr'], drop_first=True)


# Set panel structure

df_clean = df_clean.set_index(['depc', 'year'])


# Explanatory variables

exog_vars = (
    ['post_mortality', 'rural', 'agebrd', 'agegrd'] +
    [c for c in df_clean.columns if c.startswith('clgr_')]
)

exog = df_clean[exog_vars]
endog = df_clean['classdiff']


# Add constant (optional)

exog = exog.assign(const=1)


# Run DiD with FE

model = PanelOLS(
    endog,
    exog,
    entity_effects=True,   # Department FE
    time_effects=True      # Year FE
)

results = model.fit(
    cov_type='clustered',
    cluster_entity=True
)

print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:              classdiff   R-squared:                        0.3652
Estimator:                   PanelOLS   R-squared (Between):              0.2154
No. Observations:                3117   R-squared (Within):               0.3627
Date:                Wed, Apr 15 2026   R-squared (Overall):              0.3281
Time:                        20:34:04   Log-likelihood                   -4965.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      172.87
Entities:                          86   P-value                           0.0000
Avg Obs:                       36.244   Distribution:                 F(10,3005)
Min Obs:                       1.0000                                           
Max Obs:                       889.00   F-statistic (robust):             216.39
                            

In [3]:
# Cross-moment

import pandas as pd
import numpy as np


# Load data

df = pd.read_stata("marriage_level.dta")


# Convert class variables

df['clbr'] = pd.to_numeric(df['clbr'], errors='coerce')
df['clgr'] = pd.to_numeric(df['clgr'], errors='coerce')


# Dependent variable

df['classdiff'] = df['clbr'] - df['clgr']


# Drop missing values

cols_needed = ['classdiff', 'mortality', 'depc', 'year']
df_clean = df.dropna(subset=cols_needed)


# Define pre/post windows

pre_window = df_clean[(df_clean['year'] >= 1910) & (df_clean['year'] <= 1918)]
post_window = df_clean[(df_clean['year'] >= 1919) & (df_clean['year'] <= 1925)]


# Aggregate outcomes by department

pre_agg = pre_window.groupby('depc')['classdiff'].mean().reset_index()
post_agg = post_window.groupby('depc')['classdiff'].mean().reset_index()


# Merge (balanced departments)

merged = pd.merge(pre_agg, post_agg, on='depc', suffixes=('_pre', '_post'))


# Treatment: mortality (NOT post_mortality)

mortality = (
    df_clean[['depc', 'mortality']]
    .drop_duplicates(subset='depc')
)

merged = pd.merge(merged, mortality, on='depc')


# Construct variables

Z = merged['classdiff_pre'].to_numpy(dtype=np.float64)   # pre outcome
Y = merged['classdiff_post'].to_numpy(dtype=np.float64)  # post outcome
D = merged['mortality'].to_numpy(dtype=np.float64)       # treatment intensity

# Drop NaNs

mask = ~(np.isnan(Z) | np.isnan(Y) | np.isnan(D))
Z, Y, D = Z[mask], Y[mask], D[mask]


# Mean-center variables

Z -= np.mean(Z)
Y -= np.mean(Y)
D -= np.mean(D)


# Diagnostics

print("Z var:", np.var(Z), "D var:", np.var(D), "Y var:", np.var(Y))


# Cross-moment functions

def compute_ratio(Z, D, deg=2):
    var_u = np.mean(Z * D)
    sign = np.sign(var_u)

    diff_normal_D = np.mean(D**deg * Z) - deg * var_u * np.mean(D**(deg - 1))
    diff_normal_Z = np.mean(Z**deg * D) - deg * var_u * np.mean(Z**(deg - 1))

    epsilon = 1e-8
    alpha_sq = diff_normal_D / (diff_normal_Z + epsilon)

    if alpha_sq < 0:
        alpha_sq = -(abs(alpha_sq) ** (1 / (deg - 1)))
    else:
        alpha_sq = alpha_sq ** (1 / (deg - 1))

    alpha_sq = abs(alpha_sq) * sign
    return alpha_sq


def get_beta(Z, D, Y, deg=2):
    denominator = 0

    while denominator == 0:
        alpha_sq = compute_ratio(Z, D, deg)

        numerator = np.mean(D * Y) - alpha_sq * np.mean(Y * Z)
        denominator = np.mean(D * D) - alpha_sq * np.mean(D * Z)

        deg += 1
        if deg > 10:
            raise ValueError("Cannot compute beta, check data")

    return numerator / denominator



# Estimate beta

beta = get_beta(Z, D, Y, deg=2)
print(f"Beta estimated by cross-moment method (classdiff): {beta:.4f}")

Z var: 0.5101454007372196 D var: 6.8365717585925525 Y var: 0.27298149770723695
Beta estimated by cross-moment method (classdiff): -0.0044
